In [49]:
!pip install evaluate
!pip install datasets==2.21.0

In [50]:
from transformers import BertTokenizer, BertModel
from datasets import load_dataset
from evaluate import load
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"
#  You can install and import any other libraries if needed

In [51]:
# Some Chinese punctuations will be tokenized as [UNK], so we replace them with English ones
token_replacement = [
    ["：" , ":"],
    ["，" , ","],
    ["“" , "\""],
    ["”" , "\""],
    ["？" , "?"],
    ["……" , "..."],
    ["！" , "!"]
]

In [52]:

tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-uncased", cache_dir="./cache/")

In [53]:
class SemevalDataset(Dataset):
    def __init__(self, split="train") -> None:
        super().__init__()
        assert split in ["train", "validation", "test"]
        self.data = load_dataset(
            "sem_eval_2014_task_1", split=split, trust_remote_code=True, cache_dir="./cache/"
        ).to_list()

    def __getitem__(self, index):
        d = self.data[index]
        # Replace Chinese punctuations with English ones
        for k in ["premise", "hypothesis"]:
            for tok in token_replacement:
                d[k] = d[k].replace(tok[0], tok[1])
        return d

    def __len__(self):
        return len(self.data)

data_sample = SemevalDataset(split="train").data[:3]
print(f"Dataset example: \n{data_sample[0]} \n{data_sample[1]} \n{data_sample[2]}")

Dataset example: 
{'sentence_pair_id': 1, 'premise': 'A group of kids is playing in a yard and an old man is standing in the background', 'hypothesis': 'A group of boys in a yard is playing and a man is standing in the background', 'relatedness_score': 4.5, 'entailment_judgment': 0} 
{'sentence_pair_id': 2, 'premise': 'A group of children is playing in the house and there is no man standing in the background', 'hypothesis': 'A group of kids is playing in a yard and an old man is standing in the background', 'relatedness_score': 3.200000047683716, 'entailment_judgment': 0} 
{'sentence_pair_id': 3, 'premise': 'The young boys are playing outdoors and the man is smiling nearby', 'hypothesis': 'The kids are playing outdoors near a man with a smile', 'relatedness_score': 4.699999809265137, 'entailment_judgment': 1}


In [54]:
# Define the hyperparameters
# You can modify these values if needed
lr = 3e-5
epochs = 3
train_batch_size = 8
validation_batch_size = 8

In [55]:
# TODO1: Create batched data for DataLoader
# `collate_fn` is a function that defines how the data batch should be packed.
# This function will be called in the DataLoader to pack the data batch.

def collate_fn(batch):
    # Reference:　https://plainenglish.io/blog/understanding-collate-fn-in-pytorch-f9d1742647d3?utm_source=chatgpt.com
    # TODO1-1: Implement the collate_fn function
    # Write your code here
    # The input parameter is a data batch (tuple), and this function packs it into tensors.
    premises = [b["premise"] for b in batch]
    hypotheses = [b["hypothesis"] for b in batch]
    relatedness_score = torch.tensor([b["relatedness_score"] for b in batch], dtype=torch.float)
    entailment_judgment = torch.tensor([b["entailment_judgment"] for b in batch], dtype=torch.long)
    # Use tokenizer to pack tokenize and pack the data and its corresponding labels.
    input_text = tokenizer(premises, hypotheses, padding=True, truncation=True, max_length=128, return_tensors="pt")
    # Return the data batch and labels for each sub-task.
    return {
        "input_ids": input_text["input_ids"],
        "token_type_ids": input_text["token_type_ids"],
        "attention_mask": input_text["attention_mask"],
        "regression": relatedness_score,
        "classification": entailment_judgment

    }

# TODO1-2: Define your DataLoader
dl_train = DataLoader(SemevalDataset(split="train"), batch_size=train_batch_size, shuffle=True, collate_fn=collate_fn)
dl_validation = DataLoader(SemevalDataset(split="validation"), batch_size=validation_batch_size, shuffle=False, collate_fn=collate_fn)
dl_test = DataLoader(SemevalDataset(split="test"), shuffle=False, collate_fn=collate_fn)

In [56]:
# print(next(iter(dl_train)))
# print(next(iter(dl_validation)))
# print(next(iter(dl_test)))

In [57]:
# TODO2: Construct your model
# Reference:　https://medium.com/%40parkermo_86729/using-pytorch-hugging-face-and-a-pretrained-bert-model-to-create-news-source-classifier-a-46dddbd4d919
class MultiLabelModel(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Write your code here
        # Define what modules you will use in the model
        # Please use "google-bert/bert-base-uncased" model (https://huggingface.co/google-bert/bert-base-uncased)
        self.bert = BertModel.from_pretrained("google-bert/bert-base-uncased")
        # Besides the base model, you may design additional architectures by incorporating linear layers, activation functions, or other neural components.
        hidden_size = self.bert.config.hidden_size
        self.dropout = torch.nn.Dropout(0.1)
        self.regression_output = torch.nn.Linear(hidden_size, 1)
        self.classification_output = torch.nn.Linear(hidden_size, 3)

        # Remark: The use of any additional pretrained language models is not permitted.
    def forward(self, **kwargs):
        # Write your code here
        # Forward pass
        input_ids = kwargs["input_ids"]
        token_type_ids = kwargs["token_type_ids"]
        attention_mask = kwargs["attention_mask"]

        outputs = self.bert(input_ids=input_ids,
                  token_type_ids=token_type_ids,
                  attention_mask=attention_mask,
                  return_dict=True)

        cls = self.dropout(outputs.last_hidden_state[:,0,:])
        reg_output = self.regression_output(cls).squeeze(-1)
        clf_output = self.classification_output(cls)

        return {"regression": reg_output, "classification": clf_output}

In [58]:
# TODO3: Define your optimizer and loss function

model = MultiLabelModel().to(device)
# TODO3-1: Define your Optimizer
optimizer = torch.optim.AdamW(params=model.parameters(), lr=lr)

# TODO3-2: Define your loss functions (you should have two)
# Write your code here
reg_loss_function = torch.nn.MSELoss()
clf_loss_function = torch.nn.CrossEntropyLoss()
# scoring functions
psr = load("pearsonr")
acc = load("accuracy")

In [59]:
best_score = 0.0
for ep in range(epochs):
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train()
    # TODO4: Write the training loop
    # Write your code here
    epochs_loss = 0.0
    for batch in pbar:
      # train your model
      input_ids = batch["input_ids"].to(device)
      token_type_ids = batch["token_type_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      regression = batch["regression"].to(device)
      classification = batch["classification"].to(device)
      # clear gradient
      optimizer.zero_grad()
      # forward pass
      outputs = model(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
      # compute loss
      reg_output = outputs["regression"]
      clf_output = outputs["classification"]
      reg_loss = reg_loss_function(reg_output, regression)
      clf_loss = clf_loss_function(clf_output, classification)
      loss = reg_loss+clf_loss
      # back-propagation
      loss.backward()
      # model optimization
      optimizer.step()

      epochs_loss += loss.item()
      pbar.set_postfix(loss=f"{loss.item():.4f}")

    pbar = tqdm(dl_validation)
    pbar.set_description(f"Validation epoch [{ep+1}/{epochs}]")
    model.eval()
    # TODO5: Write the evaluation loop
    # Write your code here
    reg_preds = []
    reg_trues = []
    clf_preds = []
    clf_trues = []
    for batch in pbar:
      input_ids = batch["input_ids"].to(device)
      token_type_ids = batch["token_type_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      regression = batch["regression"].to(device)
      classification = batch["classification"].to(device)
      # Evaluate your model
      with torch.no_grad():
        outputs = model(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        reg_output = outputs["regression"]
        clf_output = outputs["classification"]

        reg_preds.extend(reg_output.detach().cpu().tolist())
        reg_trues.extend(regression.detach().cpu().tolist())
        clf_pred = torch.argmax(clf_output, dim=1)
        clf_preds.extend(clf_pred.detach().cpu().tolist())
        clf_trues.extend(classification.detach().cpu().tolist())


    # Output all the evaluation scores (PearsonCorr, Accuracy)
    pearson_corr = psr.compute(predictions=reg_preds, references=reg_trues)["pearsonr"]
    accuracy = acc.compute(predictions=clf_preds, references=clf_trues)["accuracy"]
    # print(f"F1 Score: {f1.compute()}")
    print(f"Pearson: {pearson_corr: .4f}, accuracy: {accuracy: .4f}")

    if pearson_corr + accuracy > best_score:
        best_score = pearson_corr + accuracy
        import os
        os.makedirs("./saved_models", exist_ok=True)
        torch.save(model.state_dict(), f'./saved_models/best_model.ckpt')

Validation epoch [1/3]: 100%|██████████| 63/63 [00:01<00:00, 51.54it/s]


Pearson:  0.8497, accuracy:  0.8440


Validation epoch [2/3]: 100%|██████████| 63/63 [00:01<00:00, 46.22it/s]


Pearson:  0.8661, accuracy:  0.8560


Validation epoch [3/3]: 100%|██████████| 63/63 [00:01<00:00, 50.07it/s]

Pearson:  0.8751, accuracy:  0.8360


In [60]:
# Load the model
model = MultiLabelModel().to(device)
model.load_state_dict(torch.load(f"./saved_models/best_model.ckpt", weights_only=True))

# Test Loop
pbar = tqdm(dl_test, desc="Test")
model.eval()

# TODO6: Write the test loop
# Write your code here
# We have loaded the best model with the highest evaluation score for you
# Please implement the test loop to evaluate the model on the test dataset
# We will have 10% of the total score for the test accuracy and pearson correlation
with torch.no_grad():
  reg_preds = []
  reg_trues = []
  clf_preds = []
  clf_trues = []
  for batch in pbar:
    input_ids = batch["input_ids"].to(device)
    token_type_ids = batch["token_type_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    regression = batch["regression"].to(device)
    classification = batch["classification"].to(device)

    outputs = model(input_ids=input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
    reg_output = outputs["regression"]
    clf_output = outputs["classification"]

    reg_preds.extend(reg_output.detach().cpu().tolist())
    reg_trues.extend(regression.detach().cpu().tolist())
    clf_pred = torch.argmax(clf_output, dim=1)
    clf_preds.extend(clf_pred.detach().cpu().tolist())
    clf_trues.extend(classification.detach().cpu().tolist())

  pearson_corr = psr.compute(predictions=reg_preds, references=reg_trues)["pearsonr"]
  accuracy = acc.compute(predictions=clf_preds, references=clf_trues)["accuracy"]
  print(f"Pearson: {pearson_corr: .4f}, Accuracy: {accuracy: .4f}")

Test: 100%|██████████| 4927/4927 [01:10<00:00, 69.97it/s] 


Pearson:  0.8817, Accuracy:  0.8744
